In [ ]:
# @title 1. CONFIGURATE AND CONNECT { display-mode: "form" }
import os, warnings, logging
from google.colab import drive, auth

# Chặn Warning ngay từ đầu dòng code
warnings.filterwarnings("ignore")
logging.getLogger('google_auth_httplib2').setLevel(logging.ERROR)

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

from googleapiclient.discovery import build

# @markdown ---
# @markdown ### 📂 CONFIGURE STORAGE
STORAGE_TYPE = "MyDrive" # @param ["MyDrive", "Shared Drives"]
FOLDER_INPUT = "" # @param {type:"string"}

def get_final_path(s_type, u_input):
    if not u_input: return None
    base = "/content/drive/MyDrive" if s_type == "MyDrive" else "/content/drive/Shared drives"
    if len(u_input) > 25 and "/" not in u_input:
        auth.authenticate_user()
        service = build('drive', 'v3')
        try:
            meta = service.files().get(fileId=u_input, fields='name').execute()
            f_name = meta['name']
            path = os.popen(f'find "/content/drive" -name "{f_name}" -type d -print -quit').read().strip()
            return path if path else os.path.join(base, f_name)
        except: return os.path.join(base, u_input)
    return os.path.join(base, u_input.lstrip('/'))

FINAL_PATH = get_final_path(STORAGE_TYPE, FOLDER_INPUT)
if FINAL_PATH:
    os.makedirs(FINAL_PATH, exist_ok=True)
    print(f"✅ Folder to store download files: {FINAL_PATH}")

In [ ]:
# @title 2. SELECT CSV FILE (DOWNLOAD FILE LIST) & START DOWNLOADING { display-mode: "form" }
import requests, os, pandas as pd
from tqdm.notebook import tqdm
from google.colab import files

def run_all_process():
    target_path = globals().get('FINAL_PATH')
    if not target_path:
        print("❌ ERROR: Please run step 1 first.")
        return

    print("📂 Step 1: Select CSV file:")
    uploaded = files.upload()
    if not uploaded: return

    try:
        csv_name = list(uploaded.keys())[0]
        df = pd.read_csv(csv_name, sep=None, engine='python')
        df.columns = df.columns.str.strip().str.lower()

        col_url = 'url' if 'url' in df.columns else None
        col_name = 'filename' if 'filename' in df.columns else ('file_name' if 'file_name' in df.columns else None)

        if not col_url or not col_name:
            print(f"❌ ERROR: Column URL or FileName not found. Current columns: {list(df.columns)}")
            return
    except Exception as e:
        print(f"❌ Error reading CSV: {e}"); return

    print(f"\n🚀 Step 2: Downloading {len(df)} files to Drive...\n")
    success, skip, fail = 0, 0, 0

    for index, row in df.iterrows():
        url = str(row[col_url]).strip()
        fname = str(row[col_name]).strip()
        dest = os.path.join(target_path, fname)

        if os.path.exists(dest):
            print(f"⏩ [{index+1}/{len(df)}] Exist: {fname}")
            skip += 1
            continue

        try:
            with requests.get(url, stream=True, timeout=60) as r:
                r.raise_for_status()
                total = int(r.headers.get('content-length', 0))

                # Để leave=True để thanh tiến trình không bị mất sau khi tải xong
                with tqdm(desc=f"({index+1}/{len(df)}) {fname}",
                          total=total, unit='B', unit_scale=True, leave=True) as bar:
                    with open(dest, 'wb') as f:
                        for chunk in r.iter_content(chunk_size=1024*1024):
                            if chunk:
                                f.write(chunk)
                                bar.update(len(chunk))
                    bar.n = total
                    bar.refresh()
            success += 1
        except Exception as e:
            print(f"❌ Error on file '{fname}': {e}")
            fail += 1

    print("\n" + "="*40)
    print(f"✅ PROCESS COMPLETED!")
    print(f"✔️ Success: {success} | ⏩ Skip: {skip} | ❌ Fail: {fail}")
    print("="*40)

run_all_process()